# 04 · 계면 vs 벌크 — 온도에 따른 구조 변화

목표 (7단계):
1. **파일 불러오기** (온도별 dm4)
2. **가상 이미지**(virtual image) — 온도별 전부
3. **q로 계면/벌크 분류** — k-means/NMF/PCA **안 씀**. 구조맵(q-대비)에서 계면 선을 **기하학적으로** 잡음 → 온도별 전부
4. **계면·벌크 평균 RDF**
5. **계면 면적 vs 온도**
6. **계면·벌크의 1st·2nd peak 변화 vs 온도** (가우시안 피팅)
7. **계면·벌크의 FSDP 위치 변화 vs 온도** (가우시안 피팅)

> 분류는 `fds.localize_interface` (구조맵 = ring-DF/total, 밝기 상쇄) 로 합니다. 계면은 **얇은 소수** 픽셀이라
> 클러스터링이 못 잡지만, 구조맵에서는 **밝은 선**으로 또렷합니다(nb3 §1c). 임계값이 아니라 **선 위치화**라
> 경계로 새지 않습니다.

## 1) 파일 불러오기 & 설정

In [ ]:
import os, glob, gc
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

DATA_ROOT = "/home/jonghoonk918/Desktop/fdstem/Amorphous/In-situ/Heating-SiO"
USE_SYNTHETIC = not os.path.isdir(DATA_ROOT)

DET_BIN        = 4          # 검출기 비닝(메모리). 합성이면 1
Q_UNIT_HINT    = "1/A"
Q_PER_PX       = 0.0120     # nb1 2b 캘리브레이션 값 (비닝하면 ×DET_BIN 자동)
BEAM_RADIUS_PX = 12
# 계면 분류용 구조 링(px, 원본 검출기 기준). nb3 §1c의 best ring을 넣으세요. 비닝은 자동 반영.
IFACE_RINGS_PX = [(14, 20), (18, 26)]
DETREND        = True       # 스캔 밝기/두께 구배 제거 후 계면 선 찾기
PER_ROW        = False      # 계면이 기울었으면 True (행마다 위치 재탐색)
CFG = fds.RDFConfig(composition={"Si": 1, "O": 2}, q_int_min=0.20, q_int_max=1.50,
                    r_min=1.10, r_max=8.0, dr=0.02, damping="lorch")
FIRST_WIN = (1.45, 1.85); SECOND_WIN = (2.3, 3.4); FSDP_WIN = (0.40, 0.95)

def make_gradient_cube(scan=(50, 60), dp=(64, 64), iface_col=30, lam=6.0,
                       bulk_r=26.0, iface_r=16.0, seed=0):
    '''합성 데모: iface_col의 세로 계면(안쪽 링=계면, 바깥 링=벌크). lam=영향길이(온도↑→lam↓→계면 소멸).'''
    rng = np.random.default_rng(seed); Sy, Sx = scan; H, W = dp
    yy, xx = np.mgrid[0:H, 0:W]; cx, cy = W/2, H/2; rr = np.hypot(xx-cx, yy-cy)
    bulk  = np.exp(-(rr-bulk_r)**2/(2*3.0**2)) + 3*np.exp(-rr**2/(2*2**2))
    iface = np.exp(-(rr-iface_r)**2/(2*3.0**2)) + 3*np.exp(-rr**2/(2*2**2))
    cube = np.empty((Sy, Sx, H, W), np.float32)
    for ix in range(Sx):
        w = np.exp(-abs(ix - iface_col)/lam)
        cube[:, ix] = ((1-w)*bulk + w*iface) + 0.02*rng.standard_normal((Sy, H, W))
    return np.clip(cube, 0, None)

def rings_for(cube):
    b = max(DET_BIN, 1) if not USE_SYNTHETIC else 1
    return [(ri/b, ro/b) for ri, ro in IFACE_RINGS_PX]

def load_cube(path):
    cb = fds.load(path, Q_UNIT_HINT)
    if DET_BIN > 1: cb = fds.bin_cube_detector(cb, DET_BIN)
    return cb

# (T, loader) 목록만 만든다 — 큐브는 한 번에 하나만 메모리에 올림
if USE_SYNTHETIC:
    Ts = [300, 500, 700, 900, 1100]
    def _mk(T):
        lam = max(0.6, 6.0*(1 - (T-300)/900))          # 온도↑ → 계면 영향길이↓
        return fds.from_array(make_gradient_cube(lam=lam, seed=int(T)), q_per_px=Q_PER_PX)
    loaders = [(T, (lambda T=T: _mk(T))) for T in Ts]
else:
    files = sorted(glob.glob(os.path.join(DATA_ROOT, "*.dm4")))
    loaders = [(fds.coordinate_from_name(os.path.splitext(os.path.basename(p))[0]),
                (lambda p=p: load_cube(p))) for p in files]
    Ts = [t for t, _ in loaders]
print("USE_SYNTHETIC =", USE_SYNTHETIC, "| temperatures:", Ts)

## 처리 — 온도별 1회 통과 (2~7번이 여기서 나온 결과를 그림)

큐브는 **한 번에 하나만** 메모리에 올려서 각 온도마다: 가상이미지(2) · 계면/벌크 분류(3) · 두 상 평균
RDF(4) · 계면 면적(5) · 피크 가우시안 피팅(6·7) 을 뽑고 큐브는 버립니다.

In [ ]:
def gfit(x, y, win):
    g = fds.fit_gaussian_peak(x, y, win[0], win[1])
    return g["center"], g["fwhm"], g["success"]

results = []
beam = max(1, BEAM_RADIUS_PX // max(DET_BIN, 1))
for T, load in loaders:
    cube = load()
    md_ = cube.mean_dp()
    (cx, cy), _ = fds.find_center(md_, fds.beam_stopper_mask(md_))
    qpp = cube.calibration.q_per_px or Q_PER_PX

    vimg = fds.bright_field(cube, center=(cx, cy), radius=max(3, beam+2))   # (2) 가상 이미지

    info = fds.localize_interface(cube, center=(cx, cy), rings=rings_for(cube),
                                  detrend=DETREND, per_row=PER_ROW)          # (3) 분류
    im_mask, bk_mask = info["interface_mask"], info["bulk_mask"]

    rec = dict(T=T, vimg=np.asarray(vimg), s=info["s"], line=info["line"],
               im_mask=im_mask, bk_mask=bk_mask, width=info["width"],
               contrast=info["contrast"], area=info["interface_area"],
               Sxy=im_mask.shape)
    for name, msk in (("if", im_mask), ("bk", bk_mask)):                     # (4) 상별 RDF
        pat = fds.average_pattern(cube, msk)
        rr = fds.pattern_to_rdf(pat, qpp, CFG, center=(cx, cy), center_beam_radius=beam)
        rec[name] = rr
        rec[name+"_r1"], rec[name+"_r1w"], _ = gfit(rr.r, rr.Gr, FIRST_WIN)  # (6) 1st peak
        rec[name+"_r2"], rec[name+"_r2w"], _ = gfit(rr.r, rr.Gr, SECOND_WIN) # (6) 2nd peak
        rec[name+"_fsdp"], _, _ = gfit(rr.q_reduced, rr.phi, FSDP_WIN)       # (7) FSDP
    results.append(rec)
    print(f"  T={T:>6.0f}K  x_if={info['x_if']:>3d}  width={info['width']:.1f}px  "
          f"area={info['interface_area']:>4d}  contrast={info['contrast']:.4f}")
    del cube; gc.collect()

results.sort(key=lambda d: d["T"])
Ts = [r["T"] for r in results]
print("done:", len(results), "temperatures")

## 2) 가상 이미지 (virtual image) — 온도별 전부

In [ ]:
n = len(results); cols = min(6, n); rows = int(np.ceil(n/cols))
fig, ax = plt.subplots(rows, cols, figsize=(2.5*cols, 2.7*rows), squeeze=False)
for i, r in enumerate(results):
    a = ax[i//cols][i%cols]
    a.imshow(r["vimg"], cmap="gray"); a.set_title(f"{int(r['T'])} K", fontsize=9); a.axis("off")
for j in range(n, rows*cols): ax[j//cols][j%cols].axis("off")
fig.suptitle("virtual bright-field image vs temperature", y=1.02)
plt.tight_layout(); plt.show()

## 3) q로 계면/벌크 분류 — 온도별 전부 (k-means/NMF/PCA 없음)

구조맵(q-대비) 위에 **계면 선**(빨강)과 **분류 결과**를 겹쳐 봅니다. 빨강=계면 밴드, 파랑=벌크.

In [ ]:
fig, ax = plt.subplots(rows, cols, figsize=(2.5*cols, 2.7*rows), squeeze=False)
for i, r in enumerate(results):
    a = ax[i//cols][i%cols]
    a.imshow(r["s"], cmap="viridis")
    ov = np.zeros((*r["Sxy"], 4))
    ov[r["bk_mask"]] = (0.1, 0.4, 1, 0.35)      # bulk = blue
    ov[r["im_mask"]] = (1, 0.1, 0.1, 0.55)      # interface = red
    a.imshow(ov)
    a.plot(r["line"], np.arange(r["Sxy"][0]), "r-", lw=0.8)
    a.set_title(f"{int(r['T'])} K  w={r['width']:.0f}px", fontsize=9); a.axis("off")
for j in range(n, rows*cols): ax[j//cols][j%cols].axis("off")
fig.suptitle("interface (red) vs bulk (blue) on structural map", y=1.02)
plt.tight_layout(); plt.show()

## 4) 계면·벌크 평균 RDF

왼쪽: 온도별 **계면** RDF, 가운데: 온도별 **벌크** RDF (무지개, 오프셋). 오른쪽: 한 온도에서 계면 vs 벌크 겹침.

In [ ]:
cmT = plt.get_cmap("rainbow"); normT = plt.Normalize(min(Ts), max(Ts))
off = 0.8
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for k, (name, ttl) in enumerate([("if", "interface"), ("bk", "bulk")]):
    for j, r in enumerate(results):
        ax[k].plot(r[name].r, r[name].Gr + j*off, color=cmT(normT(r["T"])), lw=1.1)
        ax[k].text(6.05, j*off, f"{int(r['T'])}K", fontsize=7, va="center",
                   color=cmT(normT(r["T"])))
    ax[k].set_xlim(0, 6); ax[k].set_xlabel("r (Å)"); ax[k].set_ylabel("G(r) + offset")
    ax[k].set_title(f"{ttl} RDF vs T")
r0 = results[0]
ax[2].plot(r0["if"].r, r0["if"].Gr, color="crimson", label="interface")
ax[2].plot(r0["bk"].r, r0["bk"].Gr, color="navy", label="bulk")
ax[2].axhline(0, color="0.8", lw=0.8); ax[2].set_xlim(0, 6)
ax[2].set_xlabel("r (Å)"); ax[2].set_ylabel("G(r)")
ax[2].set_title(f"interface vs bulk @ {int(r0['T'])} K"); ax[2].legend()
plt.tight_layout(); plt.show()

## 5) 계면 면적 vs 온도

계면으로 분류된 스캔 위치 수(면적)와 구조 대비. 계면이 균질화되면 **면적·대비 모두 감소**합니다.

In [ ]:
areas = [r["area"] for r in results]; widths = [r["width"] for r in results]
contr = [r["contrast"] for r in results]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(Ts, areas, "o-", color="teal")
ax2 = ax[0].twinx(); ax2.plot(Ts, widths, "s--", color="gray", alpha=0.7)
ax[0].set_xlabel("temperature (K)"); ax[0].set_ylabel("interface area (# positions)", color="teal")
ax2.set_ylabel("width (px)", color="gray"); ax[0].set_title("interface area & width vs T")
ax[1].plot(Ts, contr, "^-", color="crimson")
ax[1].set_xlabel("temperature (K)"); ax[1].set_ylabel("structural contrast")
ax[1].set_title("interface contrast vs T")
plt.tight_layout(); plt.show()

## 6) 계면·벌크의 1st·2nd peak 위치 vs 온도 (가우시안 피팅)

RDF의 1st peak(SRO, Si–O ~1.6 Å) 와 2nd shell(MRO)을 가우시안으로 피팅해 위치를 추적합니다. 계면과 벌크를
비교하면 **어느 쪽이 온도에 더 민감한지** 보입니다.
> qmax가 낮아 절대값보다 **상대 변화**(계면−벌크, 온도 추세)가 신뢰 높습니다.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for name, col in [("if", "crimson"), ("bk", "navy")]:
    ax[0].plot(Ts, [r[name+"_r1"] for r in results], "o-", color=col, label=name)
    ax[1].plot(Ts, [r[name+"_r2"] for r in results], "s-", color=col, label=name)
ax[0].set_ylabel("1st peak r₁ (Å)"); ax[0].set_title("SRO: 1st peak vs T")
ax[1].set_ylabel("2nd peak r₂ (Å)"); ax[1].set_title("MRO: 2nd peak vs T")
for a in ax: a.set_xlabel("temperature (K)"); a.legend()
plt.tight_layout(); plt.show()

## 7) 계면·벌크의 FSDP 위치 vs 온도 (가우시안 피팅)

FSDP(첫 sharp 회절 피크, φ(q)의 첫 피크 = MRO 지표)를 가우시안 피팅. FSDP가 **이동/약화**하면 중거리 질서가
바뀌는 것. 계면이 온도에 따라 벌크에 가까워지면 두 곡선이 **수렴**합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, col in [("if", "crimson"), ("bk", "navy")]:
    ax.plot(Ts, [r[name+"_fsdp"] for r in results], "^-", color=col, label=name)
ax.set_xlabel("temperature (K)"); ax.set_ylabel("FSDP q (1/Å)")
ax.set_title("FSDP (MRO) position vs T — interface vs bulk"); ax.legend()
plt.tight_layout(); plt.show()

**정리** — 클러스터링 없이 **구조맵 + 선 위치화**로 계면/벌크를 나누고, 온도별로 (2) 가상이미지,
(3) 분류, (4) 두 상 RDF, (5) 계면 면적 소멸, (6) SRO·MRO 피크, (7) FSDP 를 정량화합니다.
- 계면이 **기울면** `PER_ROW=True`. 링은 nb3 §1c의 best ring(`IFACE_RINGS_PX`)으로 맞추세요.
- 계면이 온도에 따라 **이동**해도 각 온도에서 위치를 다시 찾으므로 강건합니다.
- qmax가 낮아 절대 정량(배위수)보다 **상대 변화·추세**로 해석하세요.